<a href="https://colab.research.google.com/github/SharikaTasnim23/Thesis/blob/main/Shared_backbone_Polished.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Cell 1: Environment and Libraries Setup
!pip install librosa pandas numpy tensorflow scikit-learn

import os
import re
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, f1_score

# রেজাল্টের পুনরাবৃত্তি ঠিক রাখতে সিড সেট করা
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
# জিপিইউ অপারেশনকে যতটা সম্ভব ডিটারমিনিস্টিক করা
tf.config.experimental.enable_op_determinism()

print("TensorFlow Version:", tf.__version__)
print("Cell 1: Environment is ready!")

TensorFlow Version: 2.20.0
Cell 1: Environment is ready!


In [ ]:
# Cell 2: Global Constants and Paths
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = "/content/drive/MyDrive/Wolfset_Dataset/Wolfset_Audio_files/Wolfset_Index.csv"
AUDIO_DIR = "/content/drive/MyDrive/Wolfset_Dataset/Wolfset_Audio_files"

# অডিও প্রসেসিং প্যারামিটার
SR = 22050                 # মোটর ও শিপিং নয়েজের জন্য আদর্শ স্যাম্পলিং রেট
SEGMENT_SEC = 10            # ৩ সেকেন্ডের উইন্ডো সাইজ
SEGMENT_SAMPLES = SR * SEGMENT_SEC

# ফিচার এক্সট্র্যাকশন প্যারামিটার (টোটাল ১৭৪টি ফিচার আসবে)
N_FFT = 2048               # ফ্রিকোয়েন্সি রেজোলিউশন বাড়ানোর জন্য বড় উইন্ডো
HOP_LENGTH = 512           # ৭৫% ইন্টারনাল ওভারল্যাপ
N_MELS = 128               # ১২৮টি মেল ব্যান্ড
N_MFCC = 40                # ৪০টি এমএফসিসি কোফিসিয়েন্ট
N_BANDS = 6                # ৬টি স্পেকট্রাল কনট্রাস্ট ব্যান্ড

# টোটাল ফিচার = (১২৮ মেল + ৪০ এমএফসিসি + ৬ কনট্রাস্ট) = ১৭৪ (Mean) + ১৭৪ (Std) = ৩৪৮
print(f"Target Feature Vector Size per segment: { (N_MELS + N_MFCC + N_BANDS) * 2 } (348, 1)")

Mounted at /content/drive
Target Feature Vector Size per segment: 348 (348, 1)


In [ ]:
# Cell 3: Structural File Name Parsing using Regex Pattern
FILENAME_PATTERN = r"^[AE](\d{5})T\d{2}N\d{2}(?:x\d+)?"

# ৫টি মোটরের জন্য খালি লিস্ট তৈরি
motor_targets = {f"Motor_{i}": [] for i in range(1, 6)}
valid_rows = []
base_sessions = []

df_raw = pd.read_csv(CSV_PATH)

for idx, row in df_raw.iterrows():
    filename = str(row['File Name'])
    match = re.match(FILENAME_PATTERN, filename)

    if match:
        state_digits = match.group(1) # ৫ ডিজিটের কোড এক্সট্র্যাক্ট করা (যেমন: '13201')

        for i in range(5):
            motor_targets[f"Motor_{i+1}"].append(int(state_digits[i]))

        base_session = filename.split('x')[0] # ডাটা লিক বন্ধ করতে বেস সেশন আইডি নেওয়া
        base_sessions.append(base_session)
        valid_rows.append(idx)

# পার্স করা লেবেল দিয়ে পরিচ্ছন্ন মেটাডাটা ফ্রেম তৈরি
metadata_df = df_raw.iloc[valid_rows].copy()
metadata_df['Base_Session_ID'] = base_sessions
for motor_name, labels in motor_targets.items():
    metadata_df[motor_name] = labels

print(f"Parsing complete! Valid audio files for processing: {len(metadata_df)}")

Parsing complete! Valid audio files for processing: 168


In [ ]:
 # =============================================================
# Cell 3.5: Filter Distractor Classes (Pure Background) 46 samples
# =============================================================

# আপনার মূল ডেটাফ্রেমের নাম যদি 'df' হয়ে থাকে:
df = metadata_df.copy() # Use metadata_df as the base for filtering
df = df[df['Description\n(Targets)'] != 'Absent (or disconnected)'].reset_index(drop=True)

print(f"Data filtered successfully! Remaining samples for feature extraction: {len(df)}")

Data filtered successfully! Remaining samples for feature extraction: 122


In [ ]:
# Cell 4: Audio Preprocessing and Feature Extraction (Mean + Std Version)

def filter_active_audio(y, top_db=20):
    """রেকর্ডিংয়ের শুরু/শেষ এবং মাঝখানের সম্পূর্ণ নীরব অংশ কেটে বাদ দেয়।"""
    intervals = librosa.effects.split(y, top_db=top_db)
    if len(intervals) == 0:
        return np.array([])
    return np.concatenate([y[start:end] for start, end in intervals])

def extract_combined_features(y_chunk, sr):
    """১০ সেকেন্ডের অ디오 চাঙ্ক থেকে Mean এবং Std বের করে ৩৪৮ সাইজের ভেক্টর বানায়।"""
    # ১. Mel-Spectrogram এক্সট্র্যাকশন
    mel = librosa.feature.melspectrogram(y=y_chunk, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_mean = np.mean(mel_db, axis=1, keepdims=True)
    mel_std = np.std(mel_db, axis=1, keepdims=True) # স্ট্যান্ডার্ড ডেভিয়েশন

    # ২. MFCCs এক্সট্র্যাকশন
    mfcc = librosa.feature.mfcc(y=y_chunk, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mfcc_mean = np.mean(mfcc, axis=1, keepdims=True)
    mfcc_std = np.std(mfcc, axis=1, keepdims=True) # স্ট্যান্ডার্ড ডেভিয়েশন

    # ৩. Spectral Contrast এক্সট্র্যাকশন
    contrast = librosa.feature.spectral_contrast(y=y_chunk, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_bands=N_BANDS)
    contrast_mean = np.mean(contrast[:N_BANDS], axis=1, keepdims=True)
    contrast_std = np.std(contrast[:N_BANDS], axis=1, keepdims=True) # স্ট্যান্ডার্ড ডেভিয়েশন

    # সব Mean এবং Std আলাদা আলাদা স্ট্যাক করা
    fused_mean = np.vstack([mel_mean, mfcc_mean, contrast_mean]) # (174, 1)
    fused_std = np.vstack([mel_std, mfcc_std, contrast_std])   # (174, 1)

    # চূড়ান্ত ভার্টিকাল স্ট্যাক (Mean + Std) -> (348, 1)
    fused = np.vstack([fused_mean, fused_std])
    return fused

In [ ]:
# Cell 5: Smart Feature Extraction with Auto-Save and Fast-Load Pipeline
import os
import numpy as np
import pandas as pd

# ১. ফিচার সেভ করার জন্য ড্রাইভের পাথ নির্ধারণ (Cell 2 এর AUDIO_DIR এর ভেতরেই সেভ হবে)
FEATURES_SAVE_PATH = os.path.join(os.path.dirname(AUDIO_DIR), "wolfset_348_features.npy")
METADATA_SAVE_PATH = os.path.join(os.path.dirname(AUDIO_DIR), "wolfset_segmented_metadata_348_features.csv")

# মাল্টি-টাস্ক ইমব্যালেন্স হ্যান্ডেল করার লজিক (ওভারল্যাপ ফাংশন)
def calculate_multitask_overlap(row):
    motor_states = [row[f"Motor_{i}"] for i in range(1, 6)]
    if any(state >= 3 for state in motor_states):
        return 0.90  # রেয়ার মোটর কন্ডিশন থাকলে ৯০% ওভারল্যাপ
    return 0.0

# ২. চেক করা হচ্ছে ড্রাইভে অলরেডি ফিচার ফাইল দুটি আছে কিনা
if os.path.exists(FEATURES_SAVE_PATH) and os.path.exists(METADATA_SAVE_PATH):
    print("="*60)
    print("PRE-EXTRACTED FEATURES FOUND! LOADING FROM GOOGLE DRIVE...")
    print("="*60)

    # ড্রাইভ থেকে সরাসরি লোড করা (সময় লাগবে মাত্র ২-৩ সেকেন্ড!)
    X_features = np.load(FEATURES_SAVE_PATH).astype(np.float32)
    final_meta_df = pd.read_csv(METADATA_SAVE_PATH)

    print(f"✓ Success: Loaded Feature Matrix Shape: {X_features.shape}")
    print(f"✓ Success: Loaded Segmented Metadata Rows: {len(final_meta_df)}")

else:
    print("="*60)
    print("NO PRE-SAVED FEATURES FOUND. STARTING AUDIO PROCESSING LOOP...")
    print("  (This will take time on the first run, but will save for future runs)")
    print("="*60)

    all_features = []
    segmented_metadata = []
    segment_idx = 1

    for idx, row in metadata_df.iterrows():
        filename = row['File Name']
        filepath = os.path.join(AUDIO_DIR, filename)

        if not os.path.exists(filepath):
            continue

        try:
            y, _ = librosa.load(filepath, sr=SR)
            y_active = filter_active_audio(y, top_db=20)

            if len(y_active) < SEGMENT_SAMPLES:
                continue

            overlap = calculate_multitask_overlap(row)
            step_samples = int(SEGMENT_SAMPLES * (1.5 - overlap))

            for start in range(0, len(y_active) - SEGMENT_SAMPLES + 1, step_samples):
                y_chunk = y_active[start:start + SEGMENT_SAMPLES]
                features = extract_combined_features(y_chunk, SR)

                all_features.append(features)

                record = {
                    "Segment_ID": f"SEG_{segment_idx:05d}",
                    "Source_File_ID": filename,
                    "Base_Session_ID": row['Base_Session_ID']
                }
                for i in range(1, 6):
                    record[f"Motor_{i}"] = row[f"Motor_{i}"]
                segmented_metadata.append(record)
                segment_idx += 1

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # নম্পাই অ্যারে এবং পান্ডাস ডেটাফ্রেমে রূপান্তর
    X_features = np.array(all_features).astype(np.float32)
    final_meta_df = pd.DataFrame(segmented_metadata)

    # ৩. ভবিষ্যৎ রানগুলোর সময় বাঁচানোর জন্য ড্রাইভে পাকাপাকিভাবে সেভ করে রাখা
    print("\n Saving extracted features to Google Drive for future instant-loading...")
    np.save(FEATURES_SAVE_PATH, X_features)
    final_meta_df.to_csv(METADATA_SAVE_PATH, index=False)

    print(f"✓ Feature Extraction & Saving Complete! Shape: {X_features.shape}")

PRE-EXTRACTED FEATURES FOUND! LOADING FROM GOOGLE DRIVE...
✓ Success: Loaded Feature Matrix Shape: (1665, 348, 1)
✓ Success: Loaded Segmented Metadata Rows: 1665


In [ ]:
# ============================================================
# CELL 1: Shared-Backbone Baseline Preparation
# Group-aware split + defensive shifting + train-only normalization
# Per-head class/sample weights for multi-task learning
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
from collections import Counter

# -----------------------------
# Configuration
# -----------------------------
SEED = 42
N_SPLITS = 4                 # 4-fold style split: roughly 75% train, 25% val
GROUP_COL = "Base_Session_ID"
MOTOR_COLS = [f"Motor_{i}" for i in range(1, 6)]

USE_SAMPLE_WEIGHTS = True
MAX_CLASS_WEIGHT = 5.0
MIN_CLASS_WEIGHT = 0.25
USE_SQRT_CLASS_WEIGHT = True

RESULT_DIR = "/content/drive/MyDrive/Wolfset_Dataset/Results/shared_backbone_compact_baseline"
os.makedirs(RESULT_DIR, exist_ok=True)

# -----------------------------
# Reproducibility
# -----------------------------
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

set_all_seeds(SEED)

# -----------------------------
# Basic checks
# -----------------------------
required_cols = [GROUP_COL] + MOTOR_COLS
missing_cols = [c for c in required_cols if c not in final_meta_df.columns]

if missing_cols:
    raise ValueError(f"Missing columns in final_meta_df: {missing_cols}")

# X_features should already exist from your feature extraction cell
X_global = np.asarray(X_features).astype(np.float32)

# Convert to CNN shape: (samples, features, 1)
if len(X_global.shape) == 2:
    X_global_cnn = np.expand_dims(X_global, axis=-1)
else:
    X_global_cnn = X_global

groups_global = final_meta_df[GROUP_COL].values

print("X shape:", X_global_cnn.shape)
print("Metadata rows:", len(final_meta_df))
print("Unique groups:", final_meta_df[GROUP_COL].nunique())

# -----------------------------
# Encode motor labels separately
# This avoids label-index problems if states are not perfectly 0...N-1
# -----------------------------
label_encoders = {}
Y_encoded = {}
num_classes_per_motor = []

for motor in MOTOR_COLS:
    enc = LabelEncoder()
    y_enc = enc.fit_transform(final_meta_df[motor].astype(int).values)

    label_encoders[motor] = enc
    Y_encoded[motor] = y_enc
    num_classes_per_motor.append(len(enc.classes_))

    print(f"\n{motor}")
    print("Original states:", list(enc.classes_))
    print("Encoded states :", list(range(len(enc.classes_))))
    print("Distribution   :", dict(Counter(final_meta_df[motor].astype(int).values)))

# -----------------------------
# Stratification target for multi-output setting
# Combine 5 motor states into one combo label only for splitting
# -----------------------------
combo_labels = (
    final_meta_df[MOTOR_COLS]
    .astype(str)
    .agg("_".join, axis=1)
    .values
)

combo_encoder = LabelEncoder()
combo_y = combo_encoder.fit_transform(combo_labels)

# -----------------------------
# Group-aware split
# -----------------------------
sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

train_idx, val_idx = next(
    sgkf.split(X_global_cnn, combo_y, groups=groups_global)
)

train_idx = list(train_idx)
val_idx = list(val_idx)

# -----------------------------
# Defensive group shifting
# Ensures every motor state exists in training
# -----------------------------
def defensive_group_shift(train_idx, val_idx):
    train_idx = list(train_idx)
    val_idx = list(val_idx)

    for motor in MOTOR_COLS:
        y_all = Y_encoded[motor]

        global_states = set(np.unique(y_all))
        train_states = set(np.unique(y_all[train_idx]))

        missing_states = global_states - train_states

        if len(missing_states) > 0:
            original_missing = label_encoders[motor].inverse_transform(list(missing_states))
            print(f"\nWarning: {motor} missing states in train:", list(original_missing))
            print("Applying defensive full-group shifting...")

        for missing_state in missing_states:
            state_indices = np.where(y_all == missing_state)[0]
            missing_groups = np.unique(groups_global[state_indices])

            for group in missing_groups:
                group_indices = np.where(groups_global == group)[0]

                for idx in group_indices:
                    if idx in val_idx:
                        val_idx.remove(idx)
                    if idx not in train_idx:
                        train_idx.append(idx)

    return np.array(train_idx), np.array(val_idx)

train_idx, val_idx = defensive_group_shift(train_idx, val_idx)

# -----------------------------
# Group leakage check
# -----------------------------
train_groups = set(groups_global[train_idx])
val_groups = set(groups_global[val_idx])
group_overlap = train_groups.intersection(val_groups)

print("\nGroup leakage check:")
print("Train-Val group overlap:", len(group_overlap))

if len(group_overlap) > 0:
    raise ValueError("Group leakage detected! Same Base_Session_ID appears in train and validation.")

# -----------------------------
# Train-only normalization
# -----------------------------
X_train_raw = X_global_cnn[train_idx]
X_val_raw = X_global_cnn[val_idx]

# Normalize using training data only
train_mean = X_train_raw.mean(axis=0)
train_std = X_train_raw.std(axis=0)
train_std[train_std == 0] = 1.0

X_train = (X_train_raw - train_mean) / train_std
X_val = (X_val_raw - train_mean) / train_std

# -----------------------------
# Prepare Y dictionaries for multi-output Keras model
# -----------------------------
Y_train = {
    motor: Y_encoded[motor][train_idx].astype(np.int32)
    for motor in MOTOR_COLS
}

Y_val = {
    motor: Y_encoded[motor][val_idx].astype(np.int32)
    for motor in MOTOR_COLS
}

# -----------------------------
# Support table: train and validation distribution per motor state
# -----------------------------
support_records = []

for motor in MOTOR_COLS:
    enc = label_encoders[motor]

    for encoded_state in range(len(enc.classes_)):
        original_state = enc.inverse_transform([encoded_state])[0]

        train_count = int(np.sum(Y_train[motor] == encoded_state))
        val_count = int(np.sum(Y_val[motor] == encoded_state))

        support_records.append({
            "motor": motor,
            "encoded_state": encoded_state,
            "original_state": original_state,
            "train_support": train_count,
            "val_support": val_count,
            "status": "evaluated" if val_count > 0 else "zero_val_support"
        })

support_df = pd.DataFrame(support_records)
support_df.to_csv(f"{RESULT_DIR}/train_val_support_table.csv", index=False)

print("\nTrain/Validation support table:")
print(support_df)

# -----------------------------
# Exact row-level leakage check
# This checks exact duplicate features between train and val
# Group split is still the main leakage protection
# -----------------------------
def hash_rows(X):
    X_flat = X.reshape(X.shape[0], -1)
    return set(hash(row.tobytes()) for row in X_flat)

train_hashes = hash_rows(X_train)
val_hashes = hash_rows(X_val)

duplicate_overlap = train_hashes.intersection(val_hashes)

print("\nExact feature duplicate leakage check:")
print("Exact duplicate rows between train and val:", len(duplicate_overlap))

if len(duplicate_overlap) > 0:
    raise ValueError("Exact duplicate feature leakage detected between train and validation.")

# -----------------------------
# Per-head sample weights
# Shared-backbone replacement for per-motor perfect balancing
# -----------------------------
def compute_capped_class_weights(y_train, n_classes):
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    total = counts.sum()

    weights = np.ones(n_classes, dtype=np.float32)

    for c in range(n_classes):
        if counts[c] > 0:
            weights[c] = total / (n_classes * counts[c])
        else:
            weights[c] = 0.0

    if USE_SQRT_CLASS_WEIGHT:
        weights = np.sqrt(weights)

    weights = np.clip(weights, MIN_CLASS_WEIGHT, MAX_CLASS_WEIGHT)

    return weights, counts

sample_weights = {}
weight_records = []

for motor, n_classes in zip(MOTOR_COLS, num_classes_per_motor):
    class_weights, counts = compute_capped_class_weights(Y_train[motor], n_classes)

    sample_weights[motor] = class_weights[Y_train[motor]]

    for encoded_state in range(n_classes):
        original_state = label_encoders[motor].inverse_transform([encoded_state])[0]

        weight_records.append({
            "motor": motor,
            "encoded_state": encoded_state,
            "original_state": original_state,
            "train_count": int(counts[encoded_state]),
            "class_weight": float(class_weights[encoded_state])
        })

class_weight_df = pd.DataFrame(weight_records)
class_weight_df.to_csv(f"{RESULT_DIR}/class_weights_per_motor_head.csv", index=False)

print("\nClass weights per motor head:")
print(class_weight_df)

print("\nFinal split summary:")
print("Train samples:", X_train.shape[0])
print("Val samples  :", X_val.shape[0])
print("Train groups :", len(train_groups))
print("Val groups   :", len(val_groups))
print("Input shape  :", X_train.shape[1:])
print("Classes per motor:", num_classes_per_motor)

X shape: (1665, 348, 1)
Metadata rows: 1665
Unique groups: 94

Motor_1
Original states: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Encoded states : [0, 1, 2, 3, 4, 5]
Distribution   : {np.int64(0): 1268, np.int64(1): 20, np.int64(2): 164, np.int64(3): 54, np.int64(4): 143, np.int64(5): 16}

Motor_2
Original states: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Encoded states : [0, 1, 2, 3, 4, 5]
Distribution   : {np.int64(0): 1326, np.int64(1): 20, np.int64(2): 105, np.int64(3): 49, np.int64(4): 147, np.int64(5): 18}

Motor_3
Original states: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Encoded states : [0, 1, 2, 3, 4, 5]
Distribution   : {np.int64(0): 1327, np.int64(1): 20, np.int64(2): 104, np.int64(3): 49, np.int64(4): 147, np.int64(5): 18}

Motor_4
Original states: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Encoded states : [0, 1, 2, 3, 4]
Distribution   : {

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# -----------------------------
# Loss mode switch
# -----------------------------
# Options:
# "ce"             = sparse categorical cross-entropy
# "focal"          = sparse categorical focal loss
# "custom_existing" = use your previously defined custom_losses variable
LOSS_MODE = "focal"

LEARNING_RATE = 0.001
EPOCHS = 100
BATCH_SIZE = 32

# -----------------------------
# Optional sparse focal loss
# -----------------------------
def sparse_categorical_focal_loss(gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(tf.squeeze(y_true), tf.int32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        num_classes = tf.shape(y_pred)[-1]
        y_true_one_hot = tf.one_hot(y_true, depth=num_classes)

        ce = -tf.reduce_sum(y_true_one_hot * tf.math.log(y_pred), axis=-1)
        p_t = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)

        focal_factor = tf.pow(1.0 - p_t, gamma)

        return focal_factor * ce

    return loss

# -----------------------------
# Build shared-backbone model
# -----------------------------
def build_multitask_1d_cnn(input_shape, class_counts):
    inputs = tf.keras.Input(shape=input_shape, name="Acoustic_Input")

    # Conv Block 1
    x = tf.keras.layers.Conv1D(
        64,
        kernel_size=3,
        activation="relu",
        padding="same",
        name="Conv1D_1"
    )(inputs)
    x = tf.keras.layers.BatchNormalization(name="BatchNorm_1")(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, name="MaxPool_1")(x)

    # Conv Block 2
    x = tf.keras.layers.Conv1D(
        128,
        kernel_size=3,
        activation="relu",
        padding="same",
        name="Conv1D_2"
    )(x)
    x = tf.keras.layers.BatchNormalization(name="BatchNorm_2")(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, name="MaxPool_2")(x)

    # Conv Block 3
    x = tf.keras.layers.Conv1D(
        256,
        kernel_size=3,
        activation="relu",
        padding="same",
        name="Conv1D_3"
    )(x)
    x = tf.keras.layers.BatchNormalization(name="BatchNorm_3")(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, name="MaxPool_3")(x)

    # Shared representation
    x = tf.keras.layers.Flatten(name="Flatten")(x)
    shared_features = tf.keras.layers.Dense(
        128,
        activation="relu",
        name="Shared_Dense_128"
    )(x)
    shared_features = tf.keras.layers.Dropout(
        0.5,
        name="Shared_Dropout"
    )(shared_features)

    # Five motor-wise output heads
    outputs = {}

    for i, num_classes in enumerate(class_counts):
        motor_name = f"Motor_{i+1}"

        outputs[motor_name] = tf.keras.layers.Dense(
            num_classes,
            activation="softmax",
            name=motor_name
        )(shared_features)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="Wolfset_Shared_Backbone_MultiTask_CNN"
    )

    return model

model = build_multitask_1d_cnn(
    input_shape=X_train.shape[1:],
    class_counts=num_classes_per_motor
)

# -----------------------------
# Loss selection
# -----------------------------
if LOSS_MODE == "ce":
    losses = {
        motor: "sparse_categorical_crossentropy"
        for motor in MOTOR_COLS
    }

elif LOSS_MODE == "focal":
    losses = {
        motor: sparse_categorical_focal_loss(gamma=2.0)
        for motor in MOTOR_COLS
    }

elif LOSS_MODE == "custom_existing":
    # Use this if you already have custom_losses defined earlier
    losses = custom_losses

else:
    raise ValueError("Invalid LOSS_MODE. Use 'ce', 'focal', or 'custom_existing'.")

metrics_dict = {
    motor: ["accuracy"]
    for motor in MOTOR_COLS
}

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=losses,
    metrics=metrics_dict
)

model.summary()

# -----------------------------
# Callbacks
# -----------------------------
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1
    )
]

# -----------------------------
# Train
# -----------------------------
# Ensure sample_weights are ordered correctly for Keras fit function
# Keras expects a list of sample_weight arrays in the order of model outputs
# when the y argument is a dictionary. The MOTOR_COLS list defines this order.
ordered_sample_weights = None
if USE_SAMPLE_WEIGHTS:
    ordered_sample_weights = [sample_weights[motor] for motor in MOTOR_COLS]

history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    sample_weight=ordered_sample_weights, # Use the ordered list
    verbose=1,
    shuffle=True
)

# -----------------------------
# Plot total loss curve
# -----------------------------
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="Train total loss")
plt.plot(history.history["val_loss"], label="Validation total loss")
plt.title(f"Shared Backbone Training Curve ({LOSS_MODE})")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/training_curve_{LOSS_MODE}.png", dpi=300)
plt.show()

# -----------------------------
# Predict
# -----------------------------
pred_probs = model.predict(X_val, verbose=0)

# Keras returns dict if model outputs are dict in recent TF versions;
# otherwise, it may return list. Handle both.
if isinstance(pred_probs, dict):
    pred_dict = pred_probs
else:
    pred_dict = {
        motor: pred_probs[i]
        for i, motor in enumerate(MOTOR_COLS)
    }

# -----------------------------
# Evaluate per motor head
# -----------------------------
eval_records = []
all_correct = []

for motor in MOTOR_COLS:
    y_true = Y_val[motor]
    y_pred = np.argmax(pred_dict[motor], axis=1)

    all_labels = np.arange(len(label_encoders[motor].classes_))
    present_labels = np.unique(y_true)

    acc = accuracy_score(y_true, y_pred)

    # Macro F1 over labels actually present in validation
    macro_f1_present = f1_score(
        y_true,
        y_pred,
        labels=present_labels,
        average="macro",
        zero_division=0
    )

    # Macro F1 over all possible states, including zero-support states
    macro_f1_all = f1_score(
        y_true,
        y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=present_labels,
        average="weighted",
        zero_division=0
    )

    eval_records.append({
        "loss_mode": LOSS_MODE,
        "motor": motor,
        "accuracy": acc,
        "macro_f1_present_val_states": macro_f1_present,
        "macro_f1_all_states": macro_f1_all,
        "weighted_f1_present_val_states": weighted_f1,
        "val_support_total": len(y_true)
    })

    all_correct.append(y_true == y_pred)

    target_names = [
        str(s) for s in label_encoders[motor].classes_
    ]

    print("\n" + "=" * 70)
    print(f"CLASSIFICATION REPORT FOR {motor} | LOSS = {LOSS_MODE}")
    print("=" * 70)

    print(classification_report(
        y_true,
        y_pred,
        labels=all_labels,
        target_names=target_names,
        zero_division=0
    ))

    # Save classification report
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=all_labels,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )

    pd.DataFrame(report_dict).transpose().to_csv(
        f"{RESULT_DIR}/classification_report_{LOSS_MODE}_{motor}.csv"
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=all_labels
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=target_names
    )

    fig, ax = plt.subplots(figsize=(7, 7))
    disp.plot(ax=ax, values_format="d", xticks_rotation=45)
    plt.title(f"{motor} Confusion Matrix ({LOSS_MODE})")
    plt.tight_layout()
    plt.savefig(f"{RESULT_DIR}/confusion_matrix_{LOSS_MODE}_{motor}.png", dpi=300)
    plt.show()

# -----------------------------
# Exact match accuracy
# All five motor states must be correct together
# -----------------------------
all_correct_matrix = np.vstack(all_correct).T
exact_match_accuracy = np.mean(np.all(all_correct_matrix, axis=1))

eval_df = pd.DataFrame(eval_records)
eval_df.to_csv(f"{RESULT_DIR}/shared_backbone_eval_{LOSS_MODE}.csv", index=False)

exact_df = pd.DataFrame({
    "loss_mode": [LOSS_MODE],
    "exact_match_accuracy": [exact_match_accuracy],
    "epochs_trained": [len(history.history["loss"])]
})

exact_df.to_csv(f"{RESULT_DIR}/exact_match_{LOSS_MODE}.csv", index=False)

print("\n" + "=" * 70)
print("SHARED BACKBONE SUMMARY")
print("=" * 70)
print(eval_df)

print("\nExact Match Accuracy:")
print(exact_df)

print("\nSaved results in:")
print(RESULT_DIR)


IndentationError: unexpected indent (3027804683.py, line 329)